# Generar CSV con comparaci?n global (retiro + nota)
Rutina en pandas para crear una columna `etiqueta_comp` con la l?gica solicitada.


In [2]:
import os
import numpy as np
import pandas as pd

UMBRAL_APROBACION = 3.0

_TRUE_SET = {'1', 'true', 't', 'si', 's?', 's', 'y', 'yes'}
_FALSE_SET = {'0', 'false', 'f', 'no', 'n', ''}

def _to_bin_series(s):
    """Convierte una serie a 0/1 (float), dejando NaN si no es convertible."""
    if s is None:
        return pd.Series(dtype='float64')

    # Booleanos
    if pd.api.types.is_bool_dtype(s):
        return s.astype('float64')

    # Num?ricos (incluye strings num?ricos)
    s_num = pd.to_numeric(s, errors='coerce')

    # Strings/objetos con valores tipo s?/no/true/false
    s_clean = s.astype(str).str.strip().str.lower()
    map_dict = {**{k: 1 for k in _TRUE_SET}, **{k: 0 for k in _FALSE_SET}}
    s_map = s_clean.map(map_dict)

    # Combina: prioriza num?rico cuando exista; si no, usa mapeo
    return s_num.where(s_num.notna(), s_map)

def _to_float_series(s):
    """Convierte notas con separador decimal ',' o '.' a float; no parseable -> NaN."""
    if s is None:
        return pd.Series(dtype='float64')

    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')

    s_str = s.astype(str).str.strip()
    # Heur?stica: si hay ',' y '.', asumir '.' miles y ',' decimal
    has_comma = s_str.str.contains(',', regex=False)
    has_dot = s_str.str.contains('.', regex=False)
    s_norm = s_str.copy()
    both = has_comma & has_dot
    s_norm = s_norm.where(~both, s_norm.str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    only_comma = has_comma & ~has_dot
    s_norm = s_norm.where(~only_comma, s_norm.str.replace(',', '.', regex=False))
    # Quita espacios
    s_norm = s_norm.str.replace(' ', '', regex=False)
    return pd.to_numeric(s_norm, errors='coerce')

    # Booleanos
    if pd.api.types.is_bool_dtype(s):
        return s.astype('float64')

    # Num?ricos
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')

    # Strings / objetos
    s_clean = s.astype('string').str.strip().str.lower()
    s_map = s_clean.replace({**{k: 1 for k in _TRUE_SET}, **{k: 0 for k in _FALSE_SET}})
    return pd.to_numeric(s_map, errors='coerce')

def generar_comparacion_csv(
    input_csv,
    col_real_retiro,
    col_pred_retiro,
    col_nota_real,
    col_nota_pred,
    output_csv=None,
    fill_missing_pred_retiro=0,
    gen_csv_flag=True,
):
    """
    Lee un CSV y genera un nuevo CSV con columna 'etiqueta_comp'.
    Devuelve (output_csv, df_resumen).
    """
    df = pd.read_csv(input_csv, sep=';')

    # Conversi?n segura de notas
    nota_real = _to_float_series(df[col_nota_real])
    nota_pred = _to_float_series(df[col_nota_pred])

    # Conversi?n segura de retiro real/pred a 0/1
    real_ret = _to_bin_series(df[col_real_retiro])
    pred_ret = _to_bin_series(df[col_pred_retiro])

    # Relleno opcional para predicciones de retiro faltantes
    if fill_missing_pred_retiro is not None:
        pred_ret = pred_ret.fillna(fill_missing_pred_retiro)

    # M?scaras de NA
    na_ret = real_ret.isna() | pred_ret.isna()
    na_pred = nota_pred.isna()
    na_real = nota_real.isna()

    # Inicializa etiquetas con NA por retiro
    etiqueta = pd.Series(index=df.index, dtype='object')
    etiqueta[na_ret] = 'NA?SinRetiro'

    # NA por nota pred
    etiqueta[~na_ret & na_pred] = 'NA?SinNotaPred'

    # NA por nota real (solo si no retiro real)
    mask_no_retiro_real = ~na_ret & (real_ret == 0)
    etiqueta[mask_no_retiro_real & na_real] = 'NA?SinNotaReal'

    # Casos v?lidos (sin NA que bloquee)
    mask_valid = etiqueta.isna()

    # Caso 1: real retiro == 1
    m_real_ret = mask_valid & (real_ret == 1)
    if m_real_ret.any():
        m_pred_ret = m_real_ret & (pred_ret == 1)
        etiqueta[m_pred_ret] = 'Retiro?PredijoRetiro'

        m_pred_no = m_real_ret & (pred_ret == 0)
        m_pred_perder = m_pred_no & (nota_pred < UMBRAL_APROBACION)
        m_pred_ganar = m_pred_no & (nota_pred >= UMBRAL_APROBACION)
        etiqueta[m_pred_perder] = 'Retiro?PredijoPerder'
        etiqueta[m_pred_ganar] = 'Retiro?PredijoGanar'

    # Caso 2: real retiro == 0
    m_real_no = mask_valid & (real_ret == 0)
    if m_real_no.any():
        m_pred_ret2 = m_real_no & (pred_ret == 1)
        etiqueta[m_pred_ret2] = 'NoRetiro?PredijoRetiro'

        m_pred_no2 = m_real_no & (pred_ret == 0)
        if m_pred_no2.any():
            apr_real = nota_real >= UMBRAL_APROBACION
            apr_pred = nota_pred >= UMBRAL_APROBACION
            m_acierto = m_pred_no2 & (apr_real == apr_pred)
            m_error = m_pred_no2 & (apr_real != apr_pred)
            etiqueta[m_acierto] = 'NoRetiro?AciertoNota'
            etiqueta[m_error] = 'NoRetiro?ErrorNota'

    # Agrega columna sin modificar otras
    df = df.copy()
    df['etiqueta_comp'] = etiqueta

    # Reubica la columna justo a la derecha de la predicci?n de nota (p.ej. Prediccion_CAT)
    if col_nota_pred in df.columns:
        cols = list(df.columns)
        if 'etiqueta_comp' in cols:
            cols.remove('etiqueta_comp')
        insert_pos = cols.index(col_nota_pred) + 1
        cols.insert(insert_pos, 'etiqueta_comp')
        df = df[cols]

    # Output
    if output_csv is None:
        base, ext = os.path.splitext(input_csv)
        output_csv = f"{base}_comparacion_{ext}"


    if gen_csv_flag:
        df.to_csv(output_csv, index=False, sep=';')

    # Resumen
    resumen = df['etiqueta_comp'].value_counts(dropna=False).to_frame('conteo')
    resumen['porcentaje'] = (resumen['conteo'] / len(df) * 100).round(2)

    print('Resumen por etiqueta:')
    print(resumen)

    return output_csv, resumen

if __name__ == '__main__':
    # Ejemplo (reemplaza los placeholders con tus columnas reales)
    #input_csv = r"Resultados_Modelo_Excel_MegaPipeline\uninion_enero_resultados.csv"
    #input_csv= r"Resultados_Modelo_v2\resultados_asig_xg_cat2026-01-29120929.csv"
    #input_csv= r"Resultados_Modelo_Excel_MegaPipeline\union_febrero_resultados.csv"
    input_csv= r"Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv"
    col_real_retiro = 'Retiro_Asignatura_Cat'
    col_pred_retiro_cat = 'Clasificacion_CAT'
    col_pred_retiro_xg = 'Clasificacion_XGB'
    col_nota_real = 'resultado_final'
    col_nota_pred_cat = 'Prediccion_CAT'
    col_nota_pred_xg='Prediccion_XGB'


    print('Comparacion XGBoost:')
    generar_comparacion_csv(
        input_csv,
        col_real_retiro,
        col_pred_retiro_xg,
        col_nota_real,
        col_nota_pred_xg,
        output_csv=None,
        fill_missing_pred_retiro=0,
        gen_csv_flag=False
    )

    print(f'\n\n Comparacion CATBoost:')
    generar_comparacion_csv(
        input_csv,
        col_real_retiro,
        col_pred_retiro_cat,
        col_nota_real,
        col_nota_pred_cat,
        output_csv=None,
        fill_missing_pred_retiro=0,
        gen_csv_flag=False
    )


Comparacion XGBoost:


C:\Users\00412\AppData\Local\Temp\ipykernel_11492\1609624208.py:78: DtypeWarning: Columns (41,42,45,47,48,51,52,54,56,58,61,63,65,66,68,70,72,74,76,78,81,83,84,87,89,90,92,95,97,99,100,103,104,113,115,119,120,122,124,127,128,131,133,134,136,138,140,142,144,148,150,152,154,156,160,163,164,168,171,172,174,177,181,183,184,191,193,195,197,198,200,202,205,206,209,210,213,214,218,220,223,224,226,229,230,232,234,237,238,240,243,245,247) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv, sep=';')


Resumen por etiqueta:
                        conteo  porcentaje
etiqueta_comp                             
NoRetiro?AciertoNota     18416       85.23
NoRetiro?ErrorNota        1466        6.78
Retiro?PredijoGanar        802        3.71
NoRetiro?PredijoRetiro     568        2.63
Retiro?PredijoRetiro       193        0.89
Retiro?PredijoPerder       162        0.75


 Comparacion CATBoost:


C:\Users\00412\AppData\Local\Temp\ipykernel_11492\1609624208.py:78: DtypeWarning: Columns (41,42,45,47,48,51,52,54,56,58,61,63,65,66,68,70,72,74,76,78,81,83,84,87,89,90,92,95,97,99,100,103,104,113,115,119,120,122,124,127,128,131,133,134,136,138,140,142,144,148,150,152,154,156,160,163,164,168,171,172,174,177,181,183,184,191,193,195,197,198,200,202,205,206,209,210,213,214,218,220,223,224,226,229,230,232,234,237,238,240,243,245,247) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv, sep=';')


Resumen por etiqueta:
                        conteo  porcentaje
etiqueta_comp                             
NoRetiro?AciertoNota     18373       85.03
NoRetiro?ErrorNota        1336        6.18
Retiro?PredijoGanar        786        3.64
NoRetiro?PredijoRetiro     741        3.43
Retiro?PredijoRetiro       300        1.39
Retiro?PredijoPerder        71        0.33
